<div dir="rtl">

# المشروع الختامي: تصميم تجربة وتحليلها

**برنامج التجارب واختبارات A/B والاستدلال السببي**، العرض يوم الأربعاء 11 نوفمبر 2026

| القسم | موعد كتابته | الحالة |
|---|---|---|
| 0. بيانات المشروع، 1. السؤال والفرضية، 2. التصميم | الأحد (اليوم الأول) | ✅ مكتمل |
| 3. حجم العينة والمدة | الإثنين (اليوم الثاني) | ✅ مكتمل |
| 4. خطة التحليل، 5. إنشاء البيانات، 6. فحوص الثقة، 7. التحليل | الثلاثاء (اليوم الثالث) | ⏳ لاحقًا |
| 8. (اختياري) بديل شبه تجريبي | الثلاثاء (اليوم الثالث) | ⏳ لاحقًا |
| 9. التقرير والتوصية | الأربعاء (اليوم الرابع) | ⏳ لاحقًا |
| 10. التسليم على GitHub | آخر اليوم الرابع | ⏳ لاحقًا |

> البيانات هنا من **محاكاة**: نحدد الأثر الحقيقي بأنفسنا، ثم نرى هل يصل التحليل إليه.

</div>

<div dir="rtl">

## 0. بيانات المشروع

</div>

<div dir="rtl">

| البند | القيمة |
|---|---|
| اسم المشروع | اختبار خصم 10% لأول طلب |
| الاسم، أو أسماء أعضاء الفريق | Raed |
| الجهة أو المنتج | تطبيق متجر إلكتروني للتوصيل (منتج افتراضي للتدريب) |

</div>

<div dir="rtl">

## 1. السؤال والفرضية

</div>

<div dir="rtl">

- **القرار:** هل نطلق «خصم 10% على أول طلب» لجميع المستخدمين الجدد في التطبيق، أم نبقي الأسعار بدون خصم؟
- **الفرضية:** إذا عرضنا خصم 10% على أول طلب للمستخدم الجديد، فسوف ترتفع نسبة المستخدمين الذين يُتمّون طلبًا خلال فترة التجربة، لأن المستخدم الجديد متردد في أول تجربة شراء، والخصم يخفض المخاطرة التي يشعر بها ويعطيه سببًا للشراء الآن.
- **ما الذي يجعلنا نغير رأينا:**
  - إذا استبعدت فترة الثقة 95% أثرًا بحجم نقطة مئوية واحدة أو أكثر في نسبة الطلب (أي الأثر أصغر من أن يغطي تكلفة الخصم).
  - إذا تحسنت نسبة الطلب لكن ساء أحد مقاييس الحماية: انخفض الربح لكل مستخدم بعد الخصم، أو ارتفعت نسبة إلغاء الطلبات، أو زاد التواصل مع الدعم.

</div>

<div dir="rtl">

## 2. التصميم

</div>

<div dir="rtl">

| البند | القيمة |
|---|---|
| وحدة العشوائية | المستخدم الجديد (معرّف الحساب)، ويُثبَّت التوزيع على كل أجهزته حتى لا يرى السعرين |
| نسبة التوزيع | 50% للنسخة A (السعر الحالي بدون خصم) و50% للنسخة B (خصم 10% على أول طلب) |
| المقياس الرئيسي | نسبة الطلب: نسبة المستخدمين الجدد الذين أتمّوا طلبًا واحدًا على الأقل خلال التجربة (`ordered`) |
| مقاييس ثانوية | قيمة الطلب لمن طلب (`order_value`)، والإيراد لكل مستخدم (`revenue`) |
| مقاييس الحماية | الربح لكل مستخدم بعد تكلفة الخصم (`profit`)، ونسبة إلغاء الطلبات (`cancelled`)، والتواصل مع الدعم (`contacted_support`) |
| شرط SUTVA | قرار كل مستخدم يتأثر بالسعر الذي يراه هو فقط. الخطر: انتشار رمز الخصم في وسائل التواصل فيستخدمه أفراد من المجموعة A، أو مشاركة الحساب داخل الأسرة، أو أن يعرف مستخدم من A بوجود خصم فيؤجل شراءه. نخفف ذلك بالتوزيع على مستوى الحساب، وتطبيق الخصم تلقائيًا على الحساب بدل رمز عام، وعدم الإعلان عن العرض خارج التطبيق أثناء التجربة |

</div>

<div dir="rtl">

## أدوات جاهزة

شغّل الخلية التالية بدون تعديل. فيها كل الدوال التي استخدمناها في الأيام الأربعة.

</div>

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import html
from IPython.display import HTML, display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler
from matplotlib import ft2font
from scipy import stats

# الإصدارات الحديثة من matplotlib تعرض العربية مباشرة، والقديمة تحتاج مكتبتين صغيرتين
NATIVE_ARABIC = hasattr(ft2font, "__libraqm_version__")
if not NATIVE_ARABIC:
    try:
        import arabic_reshaper
        from bidi.algorithm import get_display
    except ImportError:
        import subprocess
        import sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arabic-reshaper", "python-bidi"], check=True)
        import arabic_reshaper
        from bidi.algorithm import get_display

rng = np.random.default_rng(2026)

# ألوان البرنامج: A تركوازي، وB برتقالي، والقيمة الحقيقية كحلي، والطريقة الساذجة رمادي
TEAL, ORANGE, NAVY, GRAY = "#0E7490", "#E58A1F", "#0F2A3F", "#A9B4BC"
plt.rcParams.update({
    "figure.figsize": (9, 4.5), "figure.dpi": 110,
    "axes.prop_cycle": cycler(color=[TEAL, ORANGE, NAVY, GRAY]),
    "axes.edgecolor": "#56656F", "axes.labelcolor": NAVY, "text.color": NAVY,
    "xtick.color": "#3D4F5C", "ytick.color": "#3D4F5C",
    "axes.grid": True, "grid.color": "#D8E1E7", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold", "font.size": 11,
    "legend.frameon": False,
})
pd.set_option("display.precision", 4)


def ar(text):
    """يجهز النص العربي ليظهر بالشكل والاتجاه الصحيحين داخل الرسوم."""
    text = str(text)
    return text if NATIVE_ARABIC else get_display(arabic_reshaper.reshape(text))


def say(*parts, sep=" "):
    """يطبع نصًا عربيًا من اليمين إلى اليسار."""
    text = html.escape(sep.join(str(p) for p in parts))
    display(HTML(f'<div dir="rtl" style="white-space: pre-wrap; line-height: 1.7;">{text}</div>'))

import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from IPython.display import Markdown, display


def n_per_group_proportions(p_control, p_treatment, alpha=0.05, power=0.80):
    z_a, z_b = stats.norm.ppf(1 - alpha / 2), stats.norm.ppf(power)
    var = p_control * (1 - p_control) + p_treatment * (1 - p_treatment)
    return int(np.ceil((z_a + z_b) ** 2 * var / (p_treatment - p_control) ** 2))


def mde_proportions(p_control, n_per_group, alpha=0.05, power=0.80):
    z_a, z_b = stats.norm.ppf(1 - alpha / 2), stats.norm.ppf(power)
    return (z_a + z_b) * np.sqrt(2 * p_control * (1 - p_control) / n_per_group)


def duration_days(n_per_group, daily_new_users, groups=2, round_to_weeks=True):
    days = int(np.ceil(n_per_group * groups / daily_new_users))
    return int(np.ceil(days / 7) * 7) if round_to_weeks else days


def srm_check(n_a, n_b, expected_share_b=0.5, threshold=0.001):
    total = n_a + n_b
    chi2, p = stats.chisquare([n_a, n_b], f_exp=[total * (1 - expected_share_b), total * expected_share_b])
    verdict = "خلل في التوزيع (SRM)، لا تثق بالنتائج" if p < threshold else "التوزيع سليم"
    say(f"A={n_a:,}   B={n_b:,}   χ²={chi2:.2f}   p={p:.2g}   النتيجة: {verdict}")
    return p


def diff_ci(a, b, alpha=0.05):
    """الفرق بين B وA مع فترة الثقة وقيمة p (بالتقريب الطبيعي)."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    diff = b.mean() - a.mean()
    se = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
    z = stats.norm.ppf(1 - alpha / 2)
    return {"الفرق": diff, "أدنى 95%": diff - z * se, "أعلى 95%": diff + z * se, "SE": se,
            "p": 2 * stats.norm.sf(abs(diff / se))}


def decide(ci_low, ci_high, mde, guardrails_ok=True):
    if ci_high < 0:
        return "لا نطلق، لأن النسخة الجديدة أضرت بالمقياس الرئيسي."
    if ci_low > 0 and not guardrails_ok:
        return "لا نطلق الآن، لأن المقياس الرئيسي تحسن لكن أحد مقاييس الحماية ساء. نعالج المشكلة ثم نعيد التحقق."
    if ci_low > 0:
        return "نطلق، لأن الأثر إيجابي بثقة ومقاييس الحماية سليمة."
    if ci_high < mde:
        return "لا نطلق، لأننا استبعدنا وجود أثر بالحجم الذي يهمنا."
    return "لا نقرر بعد. نمدد التجربة، أو نعيد تصميمها بقوة أعلى."


say("الأدوات جاهزة")

<div dir="rtl">

## 3. حجم العينة والمدة

عدّل قيم `CONFIG` حسب مشروعك. القيم تحت عنوان **"التصميم"** تعرفها أو تقدرها من الواقع. أما القيم تحت عنوان **"المحاكاة"** فهي "الحقيقة" التي لا يعرفها المحلل في الواقع، وتحددها هنا للتدريب فقط.

</div>

In [ ]:
CONFIG = {
    "project": "اختبار خصم 10% لأول طلب",
    # ---- التصميم (تعرفه قبل التجربة) ----
    "baseline_conversion": 0.10,   # 10% من المستخدمين الجدد يطلبون حاليًا خلال أسبوعهم الأول
    "mde_abs": 0.010,              # نقطة مئوية واحدة: أقل أثر يغطي تكلفة الخصم
    "alpha": 0.05,
    "power": 0.80,
    "daily_new_users": 5_000,      # متوسط التسجيلات الجديدة يوميًا
    "treatment_share": 0.5,
    # ---- المحاكاة ("الحقيقة" المخفية) ----
    "true_lift_abs": 0.008,        # الأثر الحقيقي أصغر قليلًا من MDE: اختبار صعب للتحليل
    "order_value_mean": 120,
    "order_value_sd": 50,
    "order_value_effect": 0,
    "margin": 0.25,
    "extra_cost_per_order_b": 12,  # تكلفة الخصم في B: 10% من متوسط قيمة الطلب (120 ريال)
    "cancel_rate": 0.05, "cancel_effect": 0.000,
    "support_rate": 0.03, "support_effect": 0.000,
    "activity_strength": 0.8,
    "novelty_boost": 0.0,
    "srm_bug_share": 0.0,
    "seed": 2026,
}

In [ ]:
c = CONFIG
n_per_group = n_per_group_proportions(c["baseline_conversion"], c["baseline_conversion"] + c["mde_abs"], c["alpha"], c["power"])
days = duration_days(n_per_group, c["daily_new_users"])
say(f"المشروع: {c['project']}")
say(f"حجم العينة: {n_per_group:,} في كل مجموعة، أي {2 * n_per_group:,} مستخدم في المجموع")
say(f"المدة: {days} يومًا، وعدد الأسابيع الكاملة: {days // 7}")
n_planned = days * c["daily_new_users"] // 2
say(f"عدد المستخدمين المتوقع خلال المدة: {n_planned:,} في كل مجموعة")
for label, n_g in [("في المدة المخططة", n_planned), ("إذا اكتفينا بأسبوع واحد", 7 * c["daily_new_users"] // 2),
                   ("إذا مددنا إلى أسبوعين", 14 * c["daily_new_users"] // 2)]:
    m = mde_proportions(c["baseline_conversion"], n_g, c["alpha"], c["power"])
    say(f"{label}: أصغر أثر قابل للاكتشاف حوالي {m * 100:.2f} نقطة")

<div dir="rtl">

**قرارك في حجم العينة والمدة:** هل المدة مقبولة؟ وإذا لم تكن مقبولة، فماذا ستغير؟

</div>

<div dir="rtl">

- اخترنا 1 نقطة مئوية أثرًا أدنى مهمًا لأن الخصم يكلف حوالي 12 ريالًا في كل طلب، فالأثر الأصغر لا يغطي تكلفته.
- نحتاج **14,749 مستخدمًا في كل مجموعة** (29,498 في المجموع) لاكتشاف أثر بنقطة مئوية واحدة (من 10% إلى 11%) بقوة 80% ومستوى دلالة 5%.
- مع 5,000 مستخدم جديد يوميًا يكفي ذلك في 6 أيام، ونقرّبها إلى **7 أيام (أسبوع كامل)** حتى يشمل كل أيام الأسبوع ونتجنب انحياز يوم بعينه (مثل عطلة نهاية الأسبوع).
- خلال الأسبوع يدخل 17,500 مستخدم في كل مجموعة، فيصبح أصغر أثر قابل للاكتشاف حوالي **0.90 نقطة**، وهو أفضل من الهدف (1 نقطة).
- **القرار:** المدة مقبولة، ونلتزم بها مسبقًا ولا نوقف التجربة مبكرًا عند رؤية نتيجة دالة (تجنبًا لمشكلة التلصص). إذا ظهر أثر الحداثة في فحص الأثر اليومي، نمدد إلى أسبوعين (يصبح MDE حوالي 0.64 نقطة).

</div>

<div dir="rtl">

## 4. خطة التحليل

⏳ يُكتب هذا القسم يوم الثلاثاء (اليوم الثالث).

</div>

<div dir="rtl">

## 5. إنشاء البيانات بالمحاكاة

⏳ يُكتب هذا القسم يوم الثلاثاء (اليوم الثالث).

</div>

<div dir="rtl">

## 6. فحوص الثقة

⏳ يُكتب هذا القسم يوم الثلاثاء (اليوم الثالث).

</div>

<div dir="rtl">

## 7. التحليل

⏳ يُكتب هذا القسم يوم الثلاثاء (اليوم الثالث).

</div>

<div dir="rtl">

## 8. (اختياري) إذا لم تستطع إجراء تجربة: الفروق في الفروق

⏳ يُكتب هذا القسم يوم الثلاثاء (اليوم الثالث).

</div>

<div dir="rtl">

## 9. التقرير والتوصية

⏳ يُكتب هذا القسم يوم الأربعاء (اليوم الرابع).

</div>

<div dir="rtl">

## 10. التسليم على GitHub

⏳ يُكتب هذا القسم يوم آخر اليوم الرابع.

</div>